In [ ]:
# ==============================================================================
# Step 1: Install Required Libraries & Lato Font
# ==============================================================================
!apt-get install -q -y fonts-lato
!pip install -q metpy cartopy geopandas requests shapely pytz numpy pandas ipywidgets

# ==============================================================================
# Step 2: Import Libraries & Register Lato Font
# ==============================================================================
import io
import json
import glob
import time
import pytz
import urllib.request
import requests
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.font_manager as fm
from matplotlib.colors import LinearSegmentedColormap
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.img_tiles as cimgt
from metpy.plots import USCOUNTIES, USSTATES
from shapely.geometry import shape
from datetime import datetime
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
import textwrap

warnings.filterwarnings('ignore')

# Dynamically register Lato TTF files
for font_file in glob.glob('/usr/share/fonts/truetype/lato/*.ttf'):
    fm.fontManager.addfont(font_file)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko)',
    'Accept': 'application/json, text/plain, */*'
}

# ==============================================================================
# Step 3: Complete NWS Watch, Warning & Advisory Color Dictionary
# ==============================================================================
NWS_ALERT_COLORS = {
    # Convective & Severe
    'Tornado Warning': '#FF0000', 'Tornado Watch': '#FFFF00',
    'Severe Thunderstorm Warning': '#FFA500', 'Severe Thunderstorm Watch': '#DB7093',
    'Special Marine Warning': '#FFA500', 'Special Weather Statement': '#FFE4B5',
    # Fire Weather
    'Red Flag Warning': '#FF1493', 'Fire Weather Watch': '#FFDEAD',
    # Air Quality, Fog & Smoke
    'Air Quality Alert': '#808080', 'Dense Fog Advisory': '#708090',
    'Dense Smoke Advisory': '#F0E68C',
    # Flooding & Hydrology
    'Flash Flood Warning': '#8B0000', 'Flash Flood Watch': '#2E8B57',
    'Flood Warning': '#00FF00', 'Flood Watch': '#2E8B57', 'Flood Advisory': '#00FF7F',
    'Coastal Flood Warning': '#228B22', 'Coastal Flood Advisory': '#7CFC00',
    'Lakeshore Flood Warning': '#228B22', 'Lakeshore Flood Advisory': '#7CFC00',
    # Winter Weather
    'Blizzard Warning': '#FF4500', 'Ice Storm Warning': '#8B008B',
    'Winter Storm Warning': '#FF69B4', 'Winter Storm Watch': '#4682B4',
    'Winter Weather Advisory': '#7B68EE', 'Wind Chill Warning': '#B0C4DE',
    'Wind Chill Watch': '#5F9EA0', 'Wind Chill Advisory': '#AFEEEE',
    'Freeze Warning': '#483D8B', 'Freeze Watch': '#00FFFF',
    'Frost Advisory': '#6495ED', 'Hard Freeze Warning': '#9400D3',
    'Hard Freeze Watch': '#4169E1',
    # Heat & Wind
    'Extreme Heat Warning': '#C71585', 'Extreme Heat Watch': '#800000',
    'Heat Advisory': '#FF7F50', 'High Wind Warning': '#DAA520',
    'High Wind Watch': '#B8860B', 'Wind Advisory': '#D2B48C',
    'Extreme Wind Warning': '#FF8C00', 'Lake Wind Advisory': '#D2B48C',
    'Dust Storm Warning': '#FFE4C4', 'Blowing Dust Advisory': '#BDB76B',
    # Tropical & Marine
    'Hurricane Warning': '#DC143C', 'Hurricane Watch': '#FF00FF',
    'Tropical Storm Warning': '#B22222', 'Tropical Storm Watch': '#F08080',
    'Storm Surge Warning': '#B52634', 'Storm Surge Watch': '#DB7093',
    'Gale Warning': '#DDA0DD', 'Gale Watch': '#FFC0CB',
    'Small Craft Advisory': '#D8BFD8', 'Hazardous Seas Warning': '#D8BFD8',
    'Heavy Freezing Spray Warning': '#008080',
    # Other Hazards
    'Tsunami Warning': '#FD6347', 'Tsunami Advisory': '#D2691E',
    'Avalanche Warning': '#1E90FF', 'Avalanche Advisory': '#CD853F',
    'Civil Danger Warning': '#FFB6C1', 'Hazardous Weather Outlook': '#EEE8AA'
}

# ==============================================================================
# Step 4: Robust WFO Extents, Timezone & Tiered-City Engine
# ==============================================================================
CWA_FALLBACK_EXTENTS = {
    'ABQ': [-109.1, -103.0, 31.3, 37.0], 'ABR': [-101.0, -96.5, 44.0, 46.5],
    'AKQ': [-78.5, -75.0, 35.5, 38.5],   'ALY': [-75.5, -72.8, 41.6, 44.5],
    'AMA': [-103.1, -99.9, 34.5, 37.0],  'ARX': [-93.0, -89.5, 42.8, 45.3],
    'BGM': [-77.8, -74.5, 41.2, 43.3],   'BIS': [-104.1, -98.5, 45.9, 49.0],
    'BMX': [-88.5, -85.0, 31.8, 34.5],   'BOI': [-118.5, -114.2, 41.8, 45.3],
    'BOU': [-107.2, -102.0, 38.9, 41.1], 'BOX': [-73.5, -69.8, 41.2, 43.0],
    'BRO': [-99.5, -97.1, 25.8, 27.8],   'BTV': [-74.5, -71.5, 43.6, 45.1],
    'BUF': [-79.8, -75.8, 42.0, 44.0],   'BYZ': [-111.5, -104.0, 44.8, 47.0],
    'CAE': [-82.5, -79.5, 33.0, 35.0],   'CAR': [-70.1, -66.9, 44.8, 47.5],
    'CHS': [-81.5, -79.0, 31.8, 33.5],   'CLE': [-84.8, -80.4, 40.5, 42.0],
    'CRP': [-100.0, -96.5, 27.5, 29.2],  'CTP': [-79.5, -76.0, 39.7, 42.0],
    'CYS': [-107.5, -102.5, 41.0, 43.2], 'DDC': [-102.1, -98.8, 37.0, 39.0],
    'DLH': [-95.2, -90.0, 45.8, 49.0],   'DMX': [-95.5, -92.2, 40.6, 43.5],
    'DTX': [-84.5, -82.3, 41.7, 44.2],   'DVN': [-92.5, -89.5, 40.4, 42.5],
    'EAX': [-95.5, -92.5, 38.0, 40.6],   'EKA': [-124.5, -122.5, 38.8, 42.0],
    'EPZ': [-109.0, -104.5, 31.3, 33.8], 'EWX': [-100.8, -96.8, 28.3, 31.0],
    'FFC': [-85.5, -82.2, 32.0, 35.0],   'FGF': [-99.5, -95.2, 45.9, 49.0],
    'FGZ': [-113.0, -109.0, 33.8, 37.0], 'FSD': [-99.5, -95.0, 42.5, 44.6],
    'FWD': [-99.0, -95.5, 31.0, 34.0],   'GGW': [-108.5, -104.0, 46.8, 49.0],
    'GID': [-100.5, -98.0, 39.8, 42.0],  'GJT': [-109.1, -106.0, 37.0, 41.0],
    'GLD': [-103.0, -100.0, 38.5, 40.2], 'GRB': [-90.5, -87.0, 43.8, 46.0],
    'GRR': [-86.5, -84.2, 42.0, 44.2],   'GSP': [-83.5, -80.8, 34.2, 36.0],
    'GYX': [-72.5, -69.5, 42.8, 45.3],   'HGX': [-96.8, -94.5, 28.8, 31.2],
    'HNX': [-121.0, -118.0, 35.0, 38.0], 'HUN': [-88.2, -85.5, 34.0, 35.2],
    'ICT': [-99.0, -95.2, 37.0, 39.0],   'ILM': [-80.2, -77.5, 33.5, 35.0],
    'ILN': [-85.2, -82.2, 38.4, 40.8],   'ILX': [-91.0, -87.5, 38.6, 41.0],
    'IND': [-87.8, -84.8, 38.7, 40.6],   'IWX': [-86.8, -84.2, 40.4, 42.0],
    'JAN': [-91.5, -88.5, 31.2, 34.0],   'JAX': [-83.2, -81.0, 28.8, 31.0],
    'JKL': [-84.8, -82.2, 36.6, 38.6],   'KEY': [-83.0, -80.2, 24.3, 25.8],
    'LBF': [-102.1, -98.8, 40.0, 43.0],  'LCH': [-94.2, -91.5, 29.5, 31.5],
    'LIX': [-91.5, -88.8, 29.0, 31.2],   'LKN': [-118.5, -114.0, 38.0, 42.0],
    'LMK': [-87.0, -84.0, 36.6, 39.0],   'LOT': [-89.6, -87.1, 40.7, 42.6],
    'LOX': [-121.0, -117.8, 33.2, 35.9], 'LSX': [-92.5, -89.0, 37.6, 40.2],
    'LUB': [-103.1, -100.0, 32.8, 34.6], 'LWX': [-79.5, -76.2, 37.8, 39.8],
    'LZK': [-94.5, -90.8, 33.0, 36.5],   'MAF': [-105.0, -101.0, 29.8, 33.2],
    'MEG': [-91.2, -88.2, 33.8, 36.5],   'MFL': [-81.5, -80.0, 25.2, 27.0],
    'MFR': [-124.6, -120.5, 41.2, 43.8], 'MHX': [-78.2, -75.5, 34.5, 36.0],
    'MKX': [-90.2, -87.8, 42.5, 44.2],   'MLB': [-82.0, -80.2, 27.0, 29.2],
    'MOB': [-89.2, -86.5, 30.2, 32.2],   'MPX': [-96.2, -91.5, 43.8, 46.5],
    'MQT': [-90.5, -84.0, 45.6, 47.6],   'MRX': [-85.2, -81.8, 35.0, 37.0],
    'MSO': [-117.0, -112.2, 45.2, 49.1], 'MTR': [-123.2, -121.0, 36.0, 38.8],
    'OAX': [-98.2, -95.0, 40.0, 42.6],   'OHX': [-88.0, -85.0, 35.0, 36.8],
    'OKX': [-74.8, -71.8, 40.5, 41.6],   'OTX': [-121.5, -114.5, 45.7, 49.25],
    'OUN': [-100.0, -96.0, 33.8, 37.0],  'PAH': [-89.8, -86.8, 36.5, 38.8],
    'PBZ': [-81.5, -78.8, 39.5, 41.6],   'PDT': [-121.8, -116.8, 43.8, 46.5],
    'PHI': [-76.2, -73.8, 38.8, 41.2],   'PIH': [-115.0, -111.0, 42.0, 44.8],
    'PQR': [-124.2, -120.8, 43.7, 46.5], 'PSR': [-115.0, -110.2, 32.2, 35.1],
    'PUB': [-107.0, -102.0, 37.0, 39.2], 'RAH': [-80.5, -77.5, 34.8, 36.6],
    'REV': [-121.0, -117.8, 37.8, 41.5], 'RIW': [-111.1, -105.8, 41.0, 45.0],
    'RLX': [-83.0, -79.8, 37.2, 39.5],   'RNK': [-82.0, -78.8, 36.2, 38.2],
    'SEW': [-124.8, -120.8, 46.4, 49.1], 'SGF': [-94.8, -91.2, 36.5, 38.5],
    'SGX': [-117.8, -115.8, 32.5, 34.5], 'SHV': [-95.2, -92.0, 31.0, 34.0],
    'SJT': [-102.5, -98.8, 30.2, 33.0],  'SLC': [-114.1, -109.0, 37.0, 42.0],
    'STO': [-123.2, -119.8, 37.5, 41.0], 'TAE': [-86.2, -83.0, 29.5, 32.0],
    'TBW': [-83.2, -81.5, 26.5, 29.2],   'TFX': [-113.5, -108.5, 45.0, 49.0],
    'TOP': [-98.5, -95.2, 38.0, 40.0],   'TSA': [-96.8, -94.0, 34.5, 37.0],
    'TWX': [-113.4, -109.0, 31.3, 33.6], 'UNR': [-105.5, -101.0, 43.0, 46.0],
    'VEF': [-117.8, -113.5, 35.0, 38.2]
}

CITIES_GDF_CACHE = None
COUNTIES_GDF_CACHE = None

def get_cwa_timezone(cwa_id):
    cwa_id = cwa_id.upper()
    eastern = {'AKQ', 'ALY', 'BGM', 'BOX', 'BTV', 'BUF', 'CAE', 'CAR', 'CHS', 'CLE', 'CTP', 'DTX', 'FFC', 'GSP', 'GYX', 'ILM', 'ILN', 'IND', 'IWX', 'JAX', 'JKL', 'KEY', 'LMK', 'LWX', 'MFL', 'MHX', 'MLB', 'OKX', 'PBZ', 'PHI', 'RAH', 'RLX', 'RNK', 'TAE', 'TBW'}
    central = {'ABR', 'AMA', 'ARX', 'BIS', 'BMX', 'BRO', 'CRP', 'DDC', 'DLH', 'DMX', 'DVN', 'EAX', 'EWX', 'FGF', 'FSD', 'FWD', 'GID', 'GLD', 'GRB', 'GRR', 'HGX', 'HUN', 'ICT', 'ILX', 'JAN', 'LBF', 'LCH', 'LIX', 'LOT', 'LSX', 'LUB', 'LZK', 'MAF', 'MEG', 'MKX', 'MOB', 'MPX', 'MQT', 'OAX', 'OHX', 'OUN', 'PAH', 'SGF', 'SHV', 'SJT', 'TOP', 'TSA'}
    mountain = {'ABQ', 'BOI', 'BOU', 'BYZ', 'CYS', 'EPZ', 'GGW', 'GJT', 'MSO', 'PIH', 'PUB', 'RIW', 'SLC', 'TFX', 'UNR'}
    phoenix = {'FGZ', 'PSR', 'TWX'}
    pacific = {'EKA', 'HNX', 'LKN', 'LOX', 'MFR', 'MTR', 'OTX', 'PDT', 'PQR', 'REV', 'SEW', 'SGX', 'STO', 'VEF'}

    if cwa_id in eastern:
        return 'America/New_York'
    elif cwa_id in central:
        return 'America/Chicago'
    elif cwa_id in mountain:
        return 'America/Denver'
    elif cwa_id in phoenix:
        return 'America/Phoenix'
    elif cwa_id in pacific:
        return 'America/Los_Angeles'

    try:
        ext = CWA_FALLBACK_EXTENTS.get(cwa_id, [-121.5, -114.5, 45.7, 49.25])
        clat, clon = round((ext[2] + ext[3]) / 2, 4), round((ext[0] + ext[1]) / 2, 4)
        pts = requests.get(f"https://api.weather.gov/points/{clat},{clon}", headers=HEADERS, timeout=5).json()
        return pts.get('properties', {}).get('timeZone', 'UTC')
    except Exception:
        return 'UTC'

def get_cwa_geometry_and_extent(cwa_id):
    cwa_id = cwa_id.upper()
    print(f"Querying NOAA MapServer GeoJSON for WFO: {cwa_id}...")
    try:
        url = (f"https://mapservices.weather.noaa.gov/static/rest/services/"
               f"nws_reference_maps/nws_reference_map/MapServer/1/query?"
               f"where=cwa%3D%27{cwa_id}%27&outFields=*&f=geojson")
        cwa_gdf = gpd.read_file(url)
        if not cwa_gdf.empty:
            minx, miny, maxx, maxy = cwa_gdf.total_bounds

            width = maxx - minx
            height = maxy - miny
            center_lat_rad = np.radians((miny + maxy) / 2.0)
            cos_lat = max(np.cos(center_lat_rad), 0.1)

            visual_width = width * cos_lat
            current_visual_aspect = visual_width / max(height, 0.1)
            target_aspect = 1.40

            if current_visual_aspect < target_aspect:
                target_visual_width = height * target_aspect
                target_deg_width = target_visual_width / cos_lat
                pad_x = (target_deg_width - width) / 2.0 + (width * 0.03)
                pad_y = height * 0.04
            else:
                pad_x = width * 0.05
                pad_y = (visual_width / target_aspect - height) / 2.0 + (height * 0.03)

            return [minx - pad_x, maxx + pad_x, miny - pad_y, maxy + pad_y], cwa_gdf
    except Exception as e:
        print(f"  -> MapServer query failed ({e}), using fallback extent...")

    extent = CWA_FALLBACK_EXTENTS.get(cwa_id, [-121.5, -114.5, 45.7, 49.25])
    return extent, None

def get_guaranteed_county_cities(cwa_id, extent, cwa_gdf=None):
    global CITIES_GDF_CACHE, COUNTIES_GDF_CACHE
    try:
        if CITIES_GDF_CACHE is None:
            print("Downloading U.S. Cities database...")
            cities_url = "https://raw.githubusercontent.com/fcas80/jt_files/main/uscities.csv"
            df_cities = pd.read_csv(cities_url)
            CITIES_GDF_CACHE = gpd.GeoDataFrame(
                df_cities,
                geometry=gpd.points_from_xy(df_cities.lng, df_cities.lat),
                crs="EPSG:4326"
            )

        if COUNTIES_GDF_CACHE is None:
            print("Downloading Natural Earth counties dataset...")
            counties_url = "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_2_counties.zip"
            cgdf = gpd.read_file(counties_url)
            adm0_col = 'ADM0_A3' if 'ADM0_A3' in cgdf.columns else 'adm0_a3'
            COUNTIES_GDF_CACHE = cgdf[cgdf[adm0_col] == 'USA'].copy()

        minx, maxx, miny, maxy = extent[0], extent[1], extent[2], extent[3]
        local_cities = CITIES_GDF_CACHE.cx[minx:maxx, miny:maxy].copy()
        local_counties = COUNTIES_GDF_CACHE.cx[minx:maxx, miny:maxy].copy()

        if cwa_gdf is not None and not local_cities.empty:
            local_cities = gpd.clip(local_cities, cwa_gdf.geometry)

        if local_cities.empty or local_counties.empty:
            return gpd.GeoDataFrame()

        # TRUE AREA CALCULATION: Project to Albers Equal Area (EPSG:5070) to get area in sq km
        local_counties_proj = local_counties.to_crs("EPSG:5070")
        local_counties['county_area_sqkm'] = local_counties_proj.geometry.area / 1e6

        cities_clean = local_cities[['geometry', 'city', 'population', 'county_name']].copy()
        cities_clean['city_name'] = cities_clean['city']
        cities_clean['city_pop'] = cities_clean['population'].fillna(1000)

        # Spatial Join to counties
        joined = gpd.sjoin(cities_clean, local_counties[['geometry', 'NAME', 'county_area_sqkm']], predicate='intersects')
        county_col = 'NAME' if 'NAME' in joined.columns else 'county_name'

        # Remove duplicates and sort ALL cities by population
        candidates = joined.drop_duplicates(subset=['city_name'])
        candidates = candidates.sort_values('city_pop', ascending=False)

        center_lat_rad = np.radians((miny + maxy) / 2.0)
        cos_lat = max(np.cos(center_lat_rad), 0.1)

        # Base dimensions for collision mapping
        width = maxx - minx
        height = maxy - miny
        visual_width = width * cos_lat

        # Base vertical boundary per line of text
        base_dy_min = max(height * 0.065, 0.15)

        kept_rows = []
        kept_cities_info = []
        county_plot_counts = {}

        for idx, row in candidates.iterrows():
            pt = row.geometry
            cname = str(row['city_name'])
            county = str(row[county_col])
            pop = row['city_pop']
            area_sqkm = row.get('county_area_sqkm', 1000)

            # FORMATTING: Replace spaces with newlines for multi-line text
            display_name = cname.replace(" ", "\n")
            lines = display_name.split("\n")
            max_len = max(len(l) for l in lines)
            num_lines = len(lines)

            # ACCURATE GEOGRAPHIC DENSITY THRESHOLDS:
            # Small counties (< 2,500 sq km / ~1,000 sq miles) = 0 baseline cities
            # Medium counties (< 6,000 sq km / ~2,300 sq miles) = 1 baseline city
            # Large counties (>= 6,000 sq km) = 2 baseline cities
            if area_sqkm < 1000:
                max_cities = 0
            elif area_sqkm < 3000:
                max_cities = 1
            elif area_sqkm < 5000:
                max_cities = 2
            elif area_sqkm < 7000:
                max_cities = 3
            elif area_sqkm < 9000:
                max_cities = 4
            else:
                max_cities = 5

            # STRICT CAP: Once a county hits its max_cities limit, STOP plotting!
            # (Only allow an override if a city is a true major metro > 75,000 pop)
            current_count = county_plot_counts.get(county, 0)
            if current_count >= max_cities and pop < 75000:
                continue

            too_close = False
            for kx, ky, klen, klines in kept_cities_info:
                dx_visual = abs(pt.x - kx) * cos_lat
                dy = abs(pt.y - ky)

                avg_len = (max_len + klen) / 2.0
                required_dx = max(visual_width * 0.009 * avg_len, 0.10)

                avg_lines = (num_lines + klines) / 2.0
                required_dy = base_dy_min * avg_lines

                if dx_visual < required_dx and dy < required_dy:
                    too_close = True
                    break

            if not too_close:
                row['city_name_formatted'] = display_name
                kept_rows.append(row)
                kept_cities_info.append((pt.x, pt.y, max_len, num_lines))
                county_plot_counts[county] = county_plot_counts.get(county, 0) + 1

        result = gpd.GeoDataFrame(kept_rows, crs="EPSG:4326")
        return result.sort_values('city_pop', ascending=False)
    except Exception as e:
        print(f"  -> Smart cities fallback: {e}")
        return gpd.GeoDataFrame()

class EsriTerrain(cimgt.OSM):
    def _image_url(self, tile):
        x, y, z = tile
        return f'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}.jpg'

# ==============================================================================
# Step 5: Interactive Map Generation Function
# ==============================================================================
def get_single_plural(event):
    if event == 'ALL':
        return "Active Weather Alerts"
    if event.endswith('Watch'):
        return f"{event}es"
    elif event.endswith('Advisory'):
        return f"{event[:-1]}ies"
    elif event.endswith('Warning') or event.endswith('Statement'):
        return f"{event}s"
    elif event.endswith('Danger') or event.endswith('Outlook'):
        return f"{event}"
    else:
        return f"{event}s"

def get_plural_title(events):
    if 'ALL' in events or len(events) == 0:
        return "Current Active Weather Alerts"

    plurals = [get_single_plural(e) for e in events]
    if len(plurals) == 1:
        return f"Current {plurals[0]}"
    elif len(plurals) == 2:
        return f"Current {plurals[0]} & {plurals[1]}"
    else:
        # Dynamically list all alerts for 3+ items so textwrap correctly triggers
        return "Current " + ", ".join(plurals[:-1]) + ", & " + plurals[-1]

def get_alert_priority(event_name):
    evt = event_name.lower()
    if 'warning' in evt:
        return 3
    elif 'advisory' in evt or 'statement' in evt:
        return 2
    elif 'watch' in evt:
        return 1
    return 1

def generate_cwa_map(CWA_ID, TARGET_EVENTS):
    CWA_ID = CWA_ID.upper()

    if isinstance(TARGET_EVENTS, str):
        TARGET_EVENTS = (TARGET_EVENTS,)

    print(f"--- Generating Map for K{CWA_ID} | Product(s): {', '.join(TARGET_EVENTS)} ---")

    # 5A: Bounding Box & Office Name
    extent, cwa_gdf = get_cwa_geometry_and_extent(CWA_ID)
    center_lon = (extent[0] + extent[1]) / 2
    center_lat = (extent[2] + extent[3]) / 2

    office_name = f"{CWA_ID} Forecast Office"
    try:
        office_res = requests.get(f"https://api.weather.gov/offices/{CWA_ID}", headers=HEADERS, timeout=8).json()
        office_name = office_res.get('name', office_name)
    except Exception:
        pass

    # 5B: Fetch Active US Alerts
    try:
        response = requests.get("https://api.weather.gov/alerts/active", headers=HEADERS, timeout=15)
        alerts_data = response.json()
    except Exception:
        alerts_data = {'features': []}

    alert_features = []
    zone_cache = {}

    for feature in alerts_data.get('features', []):
        props = feature.get('properties', {})
        event = props.get('event', 'Unknown Alert')

        if 'ALL' not in TARGET_EVENTS and event not in TARGET_EVENTS:
            continue

        belongs_to_cwa = False
        params = props.get('parameters', {})
        if CWA_ID in params.get('WFO', []):
            belongs_to_cwa = True
        for awips in params.get('AWIPSidentifier', []):
            if awips.endswith(CWA_ID):
                belongs_to_cwa = True
        if f"K{CWA_ID}" in props.get('sender', '').upper():
            belongs_to_cwa = True

        if not belongs_to_cwa:
            continue

        prio = get_alert_priority(event)
        geom = feature.get('geometry')
        if geom is not None:
            alert_features.append({'event': event, 'priority': prio, 'geometry': shape(geom)})
        else:
            for zone_url in props.get('affectedZones', []):
                if zone_url not in zone_cache:
                    try:
                        time.sleep(0.1)
                        z_resp = requests.get(zone_url, headers=HEADERS, timeout=10)
                        zone_cache[zone_url] = shape(z_resp.json().get('geometry')) if z_resp.status_code == 200 else None
                    except Exception:
                        zone_cache[zone_url] = None
                if zone_cache.get(zone_url) is not None:
                    alert_features.append({'event': event, 'priority': prio, 'geometry': zone_cache[zone_url]})

    alerts_gdf = gpd.GeoDataFrame(alert_features, crs="EPSG:4326") if alert_features else gpd.GeoDataFrame(columns=['event', 'priority', 'geometry'], crs="EPSG:4326")

    # SORT ALERTS BY PRIORITY: Watches (1) -> Advisories (2) -> Warnings (3) so Warnings draw on top!
    if not alerts_gdf.empty:
        alerts_gdf = alerts_gdf.sort_values('priority', ascending=True)

    # 5C: Setup Projection & Satellite Imagery
    fig = plt.figure(figsize=(14, 11), facecolor='#00112c')
    proj = ccrs.LambertConformal(central_longitude=center_lon, central_latitude=center_lat)

    ax = fig.add_axes([0, 0.04, 1.0, 0.85], projection=proj)
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_image(EsriTerrain(), 8, zorder=0)

    # 5D: Plot CWA Alert Geometries
    legend_patches = []
    plotted_events = set()

    if not alerts_gdf.empty:
        for _, row in alerts_gdf.iterrows():
            evt = row['event']
            prio = row['priority']
            color = NWS_ALERT_COLORS.get(evt, '#00FFFF')

            z_layer = 1 + prio
            ax.add_geometries([row['geometry']], crs=ccrs.PlateCarree(),
                              facecolor=color, edgecolor='none',
                              linewidth=0, alpha=1.0, zorder=z_layer)
            if evt not in plotted_events:
                legend_patches.append(mpatches.Patch(color=color, label=evt, alpha=1.0))
                plotted_events.add(evt)
    else:
        if 'ALL' in TARGET_EVENTS:
            label_text = "No Active Weather Alerts"
        elif len(TARGET_EVENTS) == 1:
            label_text = f"No Active {get_single_plural(TARGET_EVENTS[0])}"
        elif len(TARGET_EVENTS) == 2:
            label_text = f"No Active {get_single_plural(TARGET_EVENTS[0])} or {get_single_plural(TARGET_EVENTS[1])}"
        else:
            label_text = "No Active Selected Alerts"
        legend_patches.append(mpatches.Patch(color='none', label=label_text))

    leg = ax.legend(handles=legend_patches, loc='upper left', fontsize=18, borderpad=0.8,
                    handlelength=2.0, handleheight=1.2, framealpha=0.85,
                    facecolor='#222222', edgecolor='black', labelcolor='white')
    leg.set_zorder(10)

    # 5E: Political Borders, Interstates + US Highways & Cities
    try:
        ax.add_feature(USCOUNTIES.with_scale('5m'), edgecolor='black', linewidth=0.6, zorder=5)
        ax.add_feature(USSTATES.with_scale('5m'), edgecolor='black', linewidth=1.5, zorder=6)
    except Exception:
        ax.add_feature(cfeature.STATES, edgecolor='black', linewidth=1.5, zorder=6)

    try:
        bbox_gpd = (extent[0], extent[2], extent[1], extent[3])
        roads_url = "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_roads_north_america.zip"
        roads_gdf = gpd.read_file(roads_url, bbox=bbox_gpd)

        if not roads_gdf.empty:
            # Robust Case-Insensitive Filter for Interstates and Federal US Highways
            is_interstate_or_federal = pd.Series(False, index=roads_gdf.index)

            if 'CLASS' in roads_gdf.columns:
                is_interstate_or_federal |= roads_gdf['CLASS'].astype(str).str.lower().isin(['interstate', 'federal'])
            if 'CLASS_R' in roads_gdf.columns:
                is_interstate_or_federal |= roads_gdf['CLASS_R'].astype(str).str.lower().isin(['interstate', 'federal'])
            if 'type' in roads_gdf.columns:
                is_interstate_or_federal |= roads_gdf['type'].astype(str).str.lower().isin(['freeway', 'primary', 'tollway', 'major highway'])
            if 'scalerank' in roads_gdf.columns:
                is_interstate_or_federal |= (roads_gdf['scalerank'] <= 6)

            highways = roads_gdf[is_interstate_or_federal].copy()

            if not highways.empty:
                # Black casing for contrast
                ax.add_geometries(highways.geometry, crs=ccrs.PlateCarree(),
                                  facecolor='none', edgecolor='black',
                                  linewidth=1.8, alpha=0.9, zorder=7)
                # Gold/Amber highway fill
                ax.add_geometries(highways.geometry, crs=ccrs.PlateCarree(),
                                  facecolor='none', edgecolor='#f5a623',
                                  linewidth=1.0, alpha=0.95, zorder=8)

    except Exception as e:
        print(f"  -> Roads overlay notice: {e}")

    cities_gdf = get_guaranteed_county_cities(CWA_ID, extent, cwa_gdf)
    outline = [pe.withStroke(linewidth=3.0, foreground='black')]
    if not cities_gdf.empty:
        for _, row in cities_gdf.iterrows():
            pt = row.geometry

            # Avoid placing labels under top-left legend
            minx, maxx, miny, maxy = extent[0], extent[1], extent[2], extent[3]
            if (pt.x - minx) / (maxx - minx) < 0.28 and (pt.y - miny) / (maxy - miny) > 0.82:
                continue

            # 1. RESTORE THE CITY DOTS!
            ax.plot(pt.x, pt.y, marker='o', color='red', markeredgecolor='black',
                    markersize=8, transform=ccrs.PlateCarree(), zorder=9)

            # 2. FIXED ALIGNMENT - ALWAYS BELOW THE DOT
            va_align = 'top'
            offset_y = -0.045

            # 3. APPLY THE OFFSET WITH FORMATTED (MULTI-LINE) NAME
            ax.text(pt.x, pt.y + offset_y, row.get('city_name_formatted', row['city_name']),
                    transform=ccrs.PlateCarree(),
                    fontsize=15, fontweight='bold', fontfamily='Lato', color='white', zorder=9,
                    ha='center', va=va_align, path_effects=outline)

    # 5F: Gradient Banners & Dynamic Header Info
    nws_gradient_cmap = LinearSegmentedColormap.from_list('nws_blue_grad', ['#00102b', '#002959', '#004080', '#002959', '#00102b'])
    gradient_data = np.linspace(0, 1, 500).reshape(1, -1)

    for y_pos, height in [(0.89, 0.11), (0, 0.04)]:
        ax_banner = fig.add_axes([0, y_pos, 1.0, height], zorder=20)
        ax_banner.imshow(gradient_data, aspect='auto', cmap=nws_gradient_cmap)
        ax_banner.axis('off')

    for y_pos in [0.887, 0.04]:
        ax_line = fig.add_axes([0, y_pos, 1.0, 0.003], zorder=21)
        ax_line.set_facecolor('#0066cc')
        ax_line.axis('off')

    local_tz = pytz.timezone(get_cwa_timezone(CWA_ID))
    current_time = datetime.now(local_tz).strftime('%b %d, %Y %I:%M %p %Z')

    title_text = get_plural_title(TARGET_EVENTS)
    wrapped_title = "\n".join(textwrap.wrap(title_text, width=48))
    title_font = 20 if '\n' in wrapped_title else (24 if len(title_text) > 35 else 26)

    fig.text(0.02, 0.945, wrapped_title, color='white', fontsize=title_font,
             fontweight='bold', ha='left', va='center', zorder=25)

    fig.text(0.88, 0.955, "Weather Forecast Office", color='white', fontsize=12, ha='right', zorder=25)
    fig.text(0.88, 0.925, office_name, color='white', fontsize=20, fontweight='bold', ha='right', zorder=25)
    fig.text(0.88, 0.902, f"Issued: {current_time}", color='#cccccc', fontsize=11, ha='right', zorder=25)
    fig.text(0.98, 0.012, f"weather.gov/{CWA_ID.lower()}", color='white', fontsize=12, fontweight='bold', ha='right', zorder=25)

    try:
        wiki_api_url = "https://en.wikipedia.org/w/api.php?action=query&titles=File:US-NationalWeatherService-Logo.svg&prop=imageinfo&iiprop=url&iiurlwidth=200&format=json"
        req_api = urllib.request.Request(wiki_api_url, headers=HEADERS)
        res_api = urllib.request.urlopen(req_api).read()
        data = json.loads(res_api)
        page_id = list(data['query']['pages'].keys())[0]
        dynamic_logo_url = data['query']['pages'][page_id]['imageinfo'][0]['thumburl']

        logo_img = Image.open(io.BytesIO(urllib.request.urlopen(urllib.request.Request(dynamic_logo_url, headers=HEADERS)).read()))
        ax_logo = fig.add_axes([0.90, 0.895, 0.08, 0.085], zorder=25)
        ax_logo.imshow(logo_img)
        ax_logo.axis('off')
    except Exception:
        pass

    plt.show()

# ==============================================================================
# Step 6: Interactive Widgets & Execution
# ==============================================================================
conus_cwa_list = [
    ('ABQ - Albuquerque, NM', 'ABQ'), ('ABR - Aberdeen, SD', 'ABR'),
    ('AKQ - Wakefield, VA', 'AKQ'), ('ALY - Albany, NY', 'ALY'),
    ('AMA - Amarillo, TX', 'AMA'), ('ARX - La Crosse, WI', 'ARX'),
    ('BGM - Binghamton, NY', 'BGM'), ('BIS - Bismarck, ND', 'BIS'),
    ('BMX - Birmingham, AL', 'BMX'), ('BOI - Boise, ID', 'BOI'),
    ('BOU - Denver/Boulder, CO', 'BOU'), ('BOX - Boston/Norton, MA', 'BOX'),
    ('BRO - Brownsville, TX', 'BRO'), ('BTV - Burlington, VT', 'BTV'),
    ('BUF - Buffalo, NY', 'BUF'), ('BYZ - Billings, MT', 'BYZ'),
    ('CAE - Columbia, SC', 'CAE'), ('CAR - Caribou, ME', 'CAR'),
    ('CHS - Charleston, SC', 'CHS'), ('CLE - Cleveland, OH', 'CLE'),
    ('CRP - Corpus Christi, TX', 'CRP'), ('CTP - State College, PA', 'CTP'),
    ('CYS - Cheyenne, WY', 'CYS'), ('DDC - Dodge City, KS', 'DDC'),
    ('DLH - Duluth, MN', 'DLH'), ('DMX - Des Moines, IA', 'DMX'),
    ('DTX - Detroit/Pontiac, MI', 'DTX'), ('DVN - Quad Cities, IA/IL', 'DVN'),
    ('EAX - Kansas City, MO', 'EAX'), ('EKA - Eureka, CA', 'EKA'),
    ('EPZ - El Paso, TX', 'EPZ'), ('EWX - Austin/San Antonio, TX', 'EWX'),
    ('FFC - Atlanta, GA', 'FFC'), ('FGF - Grand Forks, ND', 'FGF'),
    ('FGZ - Flagstaff, AZ', 'FGZ'), ('FSD - Sioux Falls, SD', 'FSD'),
    ('FWD - Dallas/Fort Worth, TX', 'FWD'), ('GGW - Glasgow, MT', 'GGW'),
    ('GID - Hastings, NE', 'GID'), ('GJT - Grand Junction, CO', 'GJT'),
    ('GLD - Goodland, KS', 'GLD'), ('GRB - Green Bay, WI', 'GRB'),
    ('GRR - Grand Rapids, MI', 'GRR'), ('GSP - Greenville/Spartanburg, SC', 'GSP'),
    ('GYX - Portland/Gray, ME', 'GYX'), ('HGX - Houston/Galveston, TX', 'HGX'),
    ('HNX - San Joaquin Valley, CA', 'HNX'), ('HUN - Huntsville, AL', 'HUN'),
    ('ICT - Wichita, KS', 'ICT'), ('ILM - Wilmington, NC', 'ILM'),
    ('ILN - Cincinnati/Wilmington, OH', 'ILN'), ('ILX - Lincoln, IL', 'ILX'),
    ('IND - Indianapolis, IN', 'IND'), ('IWX - Northern Indiana, IN', 'IWX'),
    ('JAN - Jackson, MS', 'JAN'), ('JAX - Jacksonville, FL', 'JAX'),
    ('JKL - Jackson, KY', 'JKL'), ('KEY - Key West, FL', 'KEY'),
    ('LBF - North Platte, NE', 'LBF'), ('LCH - Lake Charles, LA', 'LCH'),
    ('LIX - New Orleans/Baton Rouge, LA', 'LIX'), ('LKN - Elko, NV', 'LKN'),
    ('LMK - Louisville, KY', 'LMK'), ('LOT - Chicago, IL', 'LOT'),
    ('LOX - Los Angeles/Oxnard, CA', 'LOX'), ('LSX - St. Louis, MO', 'LSX'),
    ('LUB - Lubbock, TX', 'LUB'), ('LWX - Baltimore/Washington, DC', 'LWX'),
    ('LZK - Little Rock, AR', 'LZK'), ('MAF - Midland/Odessa, TX', 'MAF'),
    ('MEG - Memphis, TN', 'MEG'), ('MFL - Miami, FL', 'MFL'),
    ('MFR - Medford, OR', 'MFR'), ('MHX - Morehead City, NC', 'MHX'),
    ('MKX - Milwaukee/Sullivan, WI', 'MKX'), ('MLB - Melbourne, FL', 'MLB'),
    ('MOB - Mobile, AL', 'MOB'), ('MPX - Minneapolis/St. Paul, MN', 'MPX'),
    ('MQT - Marquette, MI', 'MQT'), ('MRX - Morristown, TN', 'MRX'),
    ('MSO - Missoula, MT', 'MSO'), ('MTR - San Francisco/Monterey, CA', 'MTR'),
    ('OAX - Omaha/Valley, NE', 'OAX'), ('OHX - Nashville, TN', 'OHX'),
    ('OKX - New York, NY', 'OKX'), ('OTX - Spokane, WA', 'OTX'),
    ('OUN - Norman/Oklahoma City, OK', 'OUN'), ('PAH - Paducah, KY', 'PAH'),
    ('PBZ - Pittsburgh, PA', 'PBZ'), ('PDT - Pendleton, OR', 'PDT'),
    ('PHI - Philadelphia/Mount Holly, NJ', 'PHI'), ('PIH - Pocatello/Idaho Falls, ID', 'PIH'),
    ('PQR - Portland, OR', 'PQR'), ('PSR - Phoenix, AZ', 'PSR'),
    ('PUB - Pueblo, CO', 'PUB'), ('RAH - Raleigh/Durham, NC', 'RAH'),
    ('REV - Reno, NV', 'REV'), ('RIW - Riverton, WY', 'RIW'),
    ('RLX - Charleston, WV', 'RLX'), ('RNK - Blacksburg, VA', 'RNK'),
    ('SEW - Seattle/Tacoma, WA', 'SEW'), ('SGF - Springfield, MO', 'SGF'),
    ('SGX - San Diego, CA', 'SGX'), ('SHV - Shreveport, LA', 'SHV'),
    ('SJT - San Angelo, TX', 'SJT'), ('SLC - Salt Lake City, UT', 'SLC'),
    ('STO - Sacramento, CA', 'STO'), ('TAE - Tallahassee, FL', 'TAE'),
    ('TBW - Tampa Bay Area, FL', 'TBW'), ('TFX - Great Falls, MT', 'TFX'),
    ('TOP - Topeka, KS', 'TOP'), ('TSA - Tulsa, OK', 'TSA'),
    ('TWX - Tucson, AZ', 'TWX'), ('UNR - Rapid City, SD', 'UNR'),
    ('VEF - Las Vegas, NV', 'VEF')
]

full_wwa_list = [
    ('All Active Alerts', 'ALL'),
    # Severe Convective
    ('Tornado Warning', 'Tornado Warning'), ('Tornado Watch', 'Tornado Watch'),
    ('Severe Thunderstorm Warning', 'Severe Thunderstorm Warning'), ('Severe Thunderstorm Watch', 'Severe Thunderstorm Watch'),
    ('Special Marine Warning', 'Special Marine Warning'), ('Special Weather Statement', 'Special Weather Statement'),
    # Fire Weather
    ('Red Flag Warning', 'Red Flag Warning'), ('Fire Weather Watch', 'Fire Weather Watch'),
    # Air Quality & Fog
    ('Air Quality Alert', 'Air Quality Alert'), ('Dense Fog Advisory', 'Dense Fog Advisory'),
    ('Dense Smoke Advisory', 'Dense Smoke Advisory'),
    # Flooding & Hydrology
    ('Flash Flood Warning', 'Flash Flood Warning'), ('Flash Flood Watch', 'Flash Flood Watch'),
    ('Flood Warning', 'Flood Warning'), ('Flood Watch', 'Flood Watch'), ('Flood Advisory', 'Flood Advisory'),
    ('Coastal Flood Warning', 'Coastal Flood Warning'), ('Coastal Flood Advisory', 'Coastal Flood Advisory'),
    ('Lakeshore Flood Warning', 'Lakeshore Flood Warning'), ('Lakeshore Flood Advisory', 'Lakeshore Flood Advisory'),
    # Winter Weather
    ('Blizzard Warning', 'Blizzard Warning'), ('Ice Storm Warning', 'Ice Storm Warning'),
    ('Winter Storm Warning', 'Winter Storm Warning'), ('Winter Storm Watch', 'Winter Storm Watch'),
    ('Winter Weather Advisory', 'Winter Weather Advisory'), ('Wind Chill Warning', 'Wind Chill Warning'),
    ('Wind Chill Watch', 'Wind Chill Watch'), ('Wind Chill Advisory', 'Wind Chill Advisory'),
    ('Freeze Warning', 'Freeze Warning'), ('Freeze Watch', 'Freeze Watch'),
    ('Frost Advisory', 'Frost Advisory'), ('Hard Freeze Warning', 'Hard Freeze Warning'),
    ('Hard Freeze Watch', 'Hard Freeze Watch'),
    # Heat & Wind
    ('Extreme Heat Warning', 'Extreme Heat Warning'), ('Extreme Heat Watch', 'Extreme Heat Watch'),
    ('Heat Advisory', 'Heat Advisory'), ('High Wind Warning', 'High Wind Warning'),
    ('High Wind Watch', 'High Wind Watch'), ('Wind Advisory', 'Wind Advisory'),
    ('Extreme Wind Warning', 'Extreme Wind Warning'), ('Lake Wind Advisory', 'Lake Wind Advisory'),
    ('Dust Storm Warning', 'Dust Storm Warning'), ('Blowing Dust Advisory', 'Blowing Dust Advisory'),
    # Tropical & Marine
    ('Hurricane Warning', 'Hurricane Warning'), ('Hurricane Watch', 'Hurricane Watch'),
    ('Tropical Storm Warning', 'Tropical Storm Warning'), ('Tropical Storm Watch', 'Tropical Storm Watch'),
    ('Storm Surge Warning', 'Storm Surge Warning'), ('Storm Surge Watch', 'Storm Surge Watch'),
    ('Gale Warning', 'Gale Warning'), ('Gale Watch', 'Gale Watch'),
    ('Small Craft Advisory', 'Small Craft Advisory'), ('Hazardous Seas Warning', 'Hazardous Seas Warning'),
    ('Heavy Freezing Spray Warning', 'Heavy Freezing Spray Warning'),
    # Other
    ('Tsunami Warning', 'Tsunami Warning'), ('Tsunami Advisory', 'Tsunami Advisory'),
    ('Avalanche Warning', 'Avalanche Warning'), ('Avalanche Advisory', 'Avalanche Advisory'),
    ('Civil Danger Warning', 'Civil Danger Warning'), ('Hazardous Weather Outlook', 'Hazardous Weather Outlook')
]

cwa_dropdown = widgets.Dropdown(
    options=conus_cwa_list,
    value='OTX',
    description='Select WFO:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)

alert_selector = widgets.SelectMultiple(
    options=full_wwa_list,
    value=('Air Quality Alert',),
    description='Select Alert(s):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='360px', height='180px')
)

print("Tip: Hold Ctrl (Windows/Linux) or Cmd (Mac) while clicking to select multiple alert products simultaneously.")
widgets.interact(generate_cwa_map, CWA_ID=cwa_dropdown, TARGET_EVENTS=alert_selector);

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  fonts-lato
0 upgraded, 1 newly installed, 0 to remove and 0 not upgraded.
Need to get 2,781 kB of archives.
After this operation, 11.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 fonts-lato all 2.015-1 [2,781 kB]
Fetched 2,781 kB in 2s (1,366 kB/s)
Selecting previously unselected package fonts-lato.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../fonts-lato_2.015-1_all.deb ...
Unpacking fonts-lato (2.015-1) ...
Setting up fonts-lato (2.015-1) ...
Processing triggers for fontconfig (2.15.0-1.1ubuntu2) ...
Tip: Hold Ctrl (Windows/Linux) or Cmd (Mac) while clicking to select multiple alert products simultaneously.


interactive(children=(Dropdown(description='Select WFO:', index=85, layout=Layout(width='320px'), options=(('A…